# Problem Restatement

# Problem
Past bond programs were structured to recommend projects in each Ward to try to
maximize the likelihood of a successful outcome in a voter election. Can the results
be provided in a map format overlaid on the City Ward Map to make sure we are
getting good representation for each Ward? This may require that we have to look at
more than the top 20 locations in the City. In mapping these
collisions/volumes/collision-volume aggregate, we would envision some sort of
heat map for demonstration purposes.
# Metrics to compute
1. Rank-sum aggregate
2. segments
3. highest volume
# To Do List
- [ ] Interactive file selection in selected folder
- [ ] Filter outliers and typos
- [ ] Calculate metrics
- [ ] Adjust locations
- [ ] Plot everything

# Data Loading & Inspection

In [21]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# The years you want to import
input_years = ['19','21','23','25','26']

# This is where we will put the dataframes
df = {}

# This creates a dictionary to hold each dataframe with the `input_year` being the key
# I know concatenating strings like this is bad practice, but I wanted better readability
pd.set_option("display.max_colwidth", None)
for year in input_years:
  dataframe = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/"+"ACOG FY" + year + ".xlsx")
  df[year] = dataframe
sorted = df['19'].sort_values('On', ascending=False)
sorted

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Loc ID,County,Community,Functional Class,On,From,To,Approach,Dir,Latitude,Longitude,Latest,Latest Date,Unnamed: 13
22,52500-244,Cleveland,Norman,Collector,Webster,Eufala,Symmes,NaN,2-WAY,35.216946,-97.445095,3552,2016-01-01,"NB 1,440vpd - 32mph / SB 1,847vpd - 30mph (12-7-18)"
15,52500-231,Cleveland,Norman,Collector,W Tecumseh Rd,Charleston Rd,Porter,NaN,2-WAY,35.261971,-97.445490,15120,2016-01-01,"EB 7,141vpd - 58mph / WB 5,382vpd - 56mph (11-16-18)"
13,52500-224,Cleveland,Norman,Collector,W Tecumseh Rd,60TH AVE NW,48TH AVE NW,BETWEEN,2-WAY,35.261872,-97.519436,3165,2017-01-01,"EB 1,572vpd - 67mph / WB 1,783vpd - 62mph (11-16-18)"
12,52500-223,Cleveland,Norman,-,W Tecumseh Rd,Castlerock Rd,36TH AVE NW,BETWEEN,2-WAY,35.261906,-97.497791,8357,2017-01-01,"EB 4,540vpd - 49mph / WB 4,576vpd - 51mph (11-16-18)"
11,52500-222,Cleveland,Norman,Collector,W Tecumseh Rd,48th Ave.NW,Castlerock Rd,NaN,2-WAY,35.261893,-97.507759,5379,2017-01-01,"EB 3,004vpd - 49mph / WB 3,022vpd - 56mph (11-16-18)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,52500-264,Cleveland,Norman,-,12th Ave NE,Rock Creek,High Meadows,NaN,2-WAY,35.243206,-97.423566,18352,2016-01-01,"NB 7,506vpd - 56mph / SB 9,100vpd - 59mph (9-14-18)"
34,52500-265,Cleveland,Norman,Principal Arterial,12th Ave NE,High Meadows,Robinson,NaN,2-WAY,35.236023,-97.423525,20651,2016-01-01,"NB 9,965vpd - 51mph / SB 9,443vpd - 49mph (9-14-18)"
32,52500-261,Cleveland,Norman,-,12th Ave NE,Morren,Alameda,NaN,2-WAY,35.219627,-97.423614,28479,2016-01-01,"NB 13,100vpd - 34mph / SB 13,430vpd - 38mph (9-14-18)"
36,52500-267,Cleveland,Norman,-,120th Ave.NE,Hilltop Dr,Alameda Dr.,NaN,2-WAY,35.235200,-97.265203,1095,2016-01-01,NB 520vpd - 58mph / SB 529vpd - 53mph (10-12-18)


In [22]:
df['19'].columns

Index(['Loc ID', 'County', 'Community', 'Functional Class', 'On', 'From', 'To',
       'Approach', 'Dir', 'Latitude', 'Longitude', 'Latest', 'Latest Date',
       'Unnamed: 13'],
      dtype='object')

# Data Preparation & Transformation

In [23]:
# Phase 1
# Rename the columns

# Columns renaming map
dictionary = {
    'Loc ID':  'loc_id',
    'LOCAL_ID': 'loc_id',
    'On':      'on_road',
    'ON_ROAD': 'on_road',
    'From': 'from',
    'FROM': 'from',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'Unnamed: 16': 'count_text',
    'Unnamed: 13': 'count_text',
    'Unnamed: 11': 'count_text',
    'LATEST_COUNT': 'count_text',
    'To': 'to',
    'TO': 'to',
    'Functional Class': 'road_type',
    'Latest': 'latest',
    'Latest Date': 'latest_date'
}

# This does the column renaming
for year in df:
    df[year] = df[year].rename(columns=dictionary)

# Phase 2
# Fix column inconsistencies




# Check if `latest` and `latest_date` exist. If they don't exist, set `latest` equal to `0`
for frame in df:
  if 'latest' not in df[frame].columns:
    df[frame]['latest'] = 0

# Find the highest VPD
for frame in df:
  df[frame]['VPD1'] = df[frame]['count_text'].str.extract(r"([\d,]+)\s*vpd")
  df[frame]['VPD2'] = df[frame]['count_text'].str.extract(r"/....([\d,]+)\s*vpd")

# Manually coerce datafame '26' to fit
df['26']['road_type'] = '-'

# Phase 3
# Reduce the dataframes to only what I need and then combine them



# A new empty dictionary. This will hold the cleaned dataframes
df_reduced = {}

# Reduce the dataframes to only the columns I need
for frame in df:
  df_reduced[frame] = df[frame][['loc_id','on_road','from','latitude','longitude','count_text','to','road_type','VPD1','VPD2']].copy()

# Remove the parenthesis
for year in df_reduced:
  df_reduced[year]['count_text'] = df_reduced[year]['count_text'].str.replace(")", "", regex=False)
  df_reduced[year]['count_text'] = df_reduced[year]['count_text'].str.replace("(", "", regex=False)
# Capture the last seven characters in `count_text`, move them to column `date` and convert it to datetime
for year in df_reduced:
  df_reduced[year]['date'] = df_reduced[year]['count_text'].str.extract(r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\s*$")[0]
  df_reduced[year]['date'] = pd.to_datetime(df_reduced[year]['date'], errors="coerce")

df_reduced['19'].head(25)
#if columns latest exists
#  compare latest with VPD1 + VP2
#  if latest > VPD1 + VPD2:
#    df[highest_vehicls_per_day] = latest
#  else:
#    df[highest_vehicls_per_day] = VPD1 + VPD2





/tmp/ipykernel_430/3776908277.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_reduced[year]['date'] = pd.to_datetime(df_reduced[year]['date'], errors="coerce")
/tmp/ipykernel_430/3776908277.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_reduced[year]['date'] = pd.to_datetime(df_reduced[year]['date'], errors="coerce")
/tmp/ipykernel_430/3776908277.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_reduced[year]['date'] = pd.to_datetime(df_reduced[year]['date'], errors="coerce")
/tmp/ipykernel_430/3776908277.py:69: UserWarning: Could not infer format, so

,loc_id,on_road,from,latitude,longitude,count_text,to,road_type,VPD1,VPD2,date
0,52500-110,Imhoff Rd W,Chautauqua,35.189588,-97.447258,"EB 2,913vpd - 37mph / WB 2,651vpd - 37mph 12-7-18",Jenkins,Collector,"2,913","2,651",2018-12-07
1,52500-118,W Indian Hills Rd,48th Ave.NW,35.290583,-97.499254,"EB 2,827vpd -65mph / WB 2,686vpd - 63mph 3-15-19",36th Ave.NW,Minor Arterial,"2,827","2,686",2019-03-15
2,52500-146,W Main St,48 th W,35.218399,-97.505958,"EB 2,671vpd - 39mph / WB 2,361vpd - 45mph 3-15-19",Willoway Dr,-,"2,671","2,361",2019-03-15
3,52500-151,E Main St,Carter,35.225586,-97.427659,"EB 3,786vpd - 45mph / WB 3,121 - 50mph 9-21-18",12 th E,-,"3,786",NaN,2018-09-21
4,52500-155,E Main St,James Garner Ave,35.221381,-97.442245,"ONE WAY street, EB 6,765vpd - 30mph 12-7-18",Peters Ave,Minor Arterial,"6,765",NaN,2018-12-07
5,52500-178,W Robinson St,36th Ave.NW,35.232913,-97.490878,"EB 10,912vpd - 42mph / WB 10,337vpd - 49mph 3-8-19",I-35,Minor Arterial,"10,912","10,337",2019-03-08
6,52500-18,Se 60th Ave Se,Alameda,35.210949,-97.353040,NB 842vpd - 56mph / SB 951vpd - 50 mph 10-26-18,Lindsey,-,842,951,2018-10-26
7,52500-181,W Robinson St,Estates Dr,35.232917,-97.498942,"EB 5,383vpd - 48mph / WB 5,170vpd - 48mph 3-8-19",36 th W,Minor Arterial,"5,383","5,170",2019-03-08
8,52500-183,W Robinson St,60th Ave.NW,35.232865,-97.525575,"EB 1,950vpd - 54mph / WB 1,575vpd - 62mph 9-28-18",Cabella Ct.,Minor Arterial,"1,950","1,575",2018-09-28
9,52500-187,W Robinson St,I-35,35.232943,-97.482685,"EB 14,007vpd - 41mph / WB 15,578vpd - 38mph 3-8-19",24th Ave.NW,Principal Arterial,"14,007","15,578",2019-03-08


# Metric Computation

In [24]:
# Extract the year



# Combine everything into one big dataframe
big_frame = pd.concat(df_reduced.values(), ignore_index=True)
big_frame['date'].isna().sum()
# Find the best volume

# Calculate road saturation

#

np.int64(3)

## Step 1: Fix the vehicle counts and keep the highest volume per location
My first `VPD1`/`VPD2` regex missed the second direction on about 12 rows (e.g. `WB 3,121 - 50mph` has no "vpd", and `NB 891 50mph` puts the speed right after the count). Those roads would have shown half their real traffic.

New approach (`get_vpd`): delete the date, delete every speed (`__mph`), and whatever numbers are left are the counts, which I add together. A typo guard handles `3,3339vpd`: it isn't a properly written number, so I assume that direction matches the other one. Counts written with a space, like `4, 436`, are kept together.

Then I keep **one row per `loc_id`, the year with the highest volume**, as the instructions require. Rows with no count (one is under construction) or no coordinates are dropped and counted.

In [ ]:
# Folder that holds the collision file and the ward GeoJSON (same as the ACOG files)
DATA_FOLDER = "/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/"

# ---------- Fix VPD: add up every count in the text ----------
import re

def get_vpd(text):
    """'EB 2,913vpd - 37mph / WB 2,651vpd - 37mph 12-7-18' -> 5564"""
    if not isinstance(text, str) or 'vp' not in text.lower():
        return None                                                   # no count in this row
    text = re.sub(r'\d{1,2}\s*[-/]\s*\d{1,2}\s*[-/]\s*\d{2,4}', ' ', text)  # 1. delete the date
    text = re.sub(r'<?\d+\s*mph', ' ', text, flags=re.IGNORECASE)          # 2. delete the speeds
    numbers = re.findall(r'\d[\d,]*(?:\s\d{3})?', text)                    # 3. what's left = the counts ('4, 436' stays together)
    counts = []
    for n in numbers:
        if re.fullmatch(r'\d{1,3}(?:,\s?\d{3})*', n):                    # a properly written number like 2,913
            counts.append(int(n.replace(',', '').replace(' ', '')))
    # Typo guard: '3,3339' is not a properly written number. If one direction is a typo,
    # assume it matches the other direction.
    if len(numbers) == 2 and len(counts) == 1:
        counts = counts * 2
    if len(counts) == 0:
        return None
    return sum(counts)

big_frame['vpd'] = big_frame['count_text'].apply(get_vpd)
big_frame['one_way'] = big_frame['count_text'].str.contains('ONE WAY', case=False, na=False)

# Two rows (FY25, FY26) have no Loc ID -> give them a made-up ID from the road names
no_id = big_frame['loc_id'].isna()
big_frame.loc[no_id, 'loc_id'] = 'NEW ' + big_frame['on_road'] + ' ' + big_frame['from'] + '-' + big_frame['to']

# Keep ONE row per location: the year with the HIGHEST volume (the instructions say to use the highest)
usable = big_frame.dropna(subset=['vpd', 'latitude', 'longitude'])
print('Rows dropped (no count or no coordinates):', len(big_frame) - len(usable))
segments = usable.sort_values('vpd', ascending=False).drop_duplicates('loc_id').reset_index(drop=True)
print('Count locations (segments):', len(segments))

## Step 2: Capacity and V/C ratio
I use the **LOS E (2025 OCARTS)** capacities. The jargon sheet defines capacity as the upper limit of flow before congestion, and that is what LOS E represents. LOS C is a lower "comfortable" service level.

**Assumption:** the number of lanes isn't in the data, and I did not check all ~200 locations on the aerial map. I estimate: one-way streets = 2 lanes, 12,000+ vpd = 4-lane undivided, otherwise 2-lane. Any location checked on the City's aerial map can be added to `LANE_OVERRIDES` and it replaces the estimate.

In [ ]:
# ---------- Capacity and V/C ----------
# ACOG 2025 OCARTS LOS E capacities (vehicles per day)
CAPACITY = {
    '2 Lane Arterial': 17100,
    '4 Lane Arterial (Undivided)': 34200,
    '4 Lane Arterial (Divided)': 38000,
    '5 Lane Arterial (Center Turn Lane)': 36000,
    '6 Lane Arterial (Undivided)': 52300,
    '6 Lane Arterial (Divided)': 58000,
    'One Way 2 Lanes': 2 * 11000,
}

# If you check a location on the aerial map, write it here and it overrides the estimate
LANE_OVERRIDES = {
    # '52500-192': '5 Lane Arterial (Center Turn Lane)',
}

def estimate_facility(row):
    """Best guess of the road type when we haven't checked the aerial map."""
    if row['loc_id'] in LANE_OVERRIDES:
        return LANE_OVERRIDES[row['loc_id']]
    if row['one_way']:
        return 'One Way 2 Lanes'
    if row['vpd'] >= 12000:
        return '4 Lane Arterial (Undivided)'
    return '2 Lane Arterial'

segments['facility'] = segments.apply(estimate_facility, axis=1)
segments['capacity'] = segments['facility'].map(CAPACITY)
segments['vc'] = (segments['vpd'] / segments['capacity']).round(2)

## Step 3: Collision locations (method from the professor's tutorial)
* Remove duplicate `AccidentNumber`s (the instructions ask for *unique* collisions)
* Remove private-property crashes and crashes with no coordinates (`-361`)
* Move "From" crashes the stated distance and direction away from the intersection (`move_point`)

In [ ]:
# ---------- Collision locations (from the professor's tutorial) ----------
import numpy as np
import math

crashes = pd.read_excel(DATA_FOLDER + 'Traffic_Project_OU_FALL_PROJECT.xlsx', sheet_name='Base_Data')
crashes = crashes.drop_duplicates('AccidentNumber')                              # unique collisions only
crashes = crashes[~crashes['StreetName'].str.contains('PRIVATE', na=False)]      # public roads only
crashes.loc[crashes['Lat'] < -90, 'Lat'] = np.nan                                # -361 means "no location"
crashes.loc[crashes['Lon'] < -180, 'Lon'] = np.nan
crashes = crashes.dropna(subset=['Lat', 'Lon']).copy()

FEET_PER_DEGREE_LAT = 364000

def move_point(lat, lon, direction, distance_ft):
    if pd.isna(distance_ft) or distance_ft > 5280:
        return (lat, lon)
    feet_per_degree_lon = FEET_PER_DEGREE_LAT * math.cos(math.radians(lat))
    if direction == 'DirectionNorth':
        lat = lat + distance_ft / FEET_PER_DEGREE_LAT
    elif direction == 'DirectionSouth':
        lat = lat - distance_ft / FEET_PER_DEGREE_LAT
    elif direction == 'DirectionEast':
        lon = lon + distance_ft / feet_per_degree_lon
    elif direction == 'DirectionWest':
        lon = lon - distance_ft / feet_per_degree_lon
    return (lat, lon)

def crash_location(row):
    if row['From_At'] == 'From':
        return move_point(row['Lat'], row['Lon'], row['Direction'], row['Distance'])
    return (row['Lat'], row['Lon'])

locations = crashes.apply(crash_location, axis=1)
crashes['crash_lat'] = [pair[0] for pair in locations]
crashes['crash_lon'] = [pair[1] for pair in locations]
print('Usable collisions:', len(crashes))

## Step 4: Assign collisions to segments
Each count location is treated as a segment. A collision belongs to a segment when:
1. it happened **on the same road**. `StreetName` is the road the crash was on, so a crash coded "48th Ave NE, 1000 ft north of Franklin" goes to 48th Ave, not Franklin, as the instructions describe.
2. it is **within half a mile** of the count location, so each segment covers about 1 mile of road.

If two count locations on the same road are close together, the crash goes to the **nearest** one, so no collision is counted twice. Collisions that don't match any segment are left out, following the instructions' hint.

In [ ]:
# ---------- Match each collision to a segment ----------
REMOVE_WORDS = ['N', 'S', 'E', 'W', 'AVE', 'ST', 'RD', 'BLVD', 'DR', 'PKWY', 'LN', 'LANE', 'HWY', 'CT', 'CIR', 'PL', 'TRL', 'WAY']
QUADRANTS = ['NE', 'NW', 'SE', 'SW']

def clean_name(name):
    """'N Flood Ave.' -> 'FLOOD'   '36th Ave.NW' -> '36TH NW'   'Se 60th Ave Se' -> '60TH SE'"""
    if not isinstance(name, str):
        return ''
    name = name.split(' / ')[0]                                 # 'N FLOOD AVE / HIGHLAND' -> 'N FLOOD AVE'
    name = name.upper().replace('.', ' ')
    name = re.sub(r'(\d+)\s+(TH|ND|RD|ST)\b', r'\1\2', name)     # '36 TH' -> '36TH'
    words = [w for w in name.split() if w not in REMOVE_WORDS]
    if len(words) > 1 and words[0] in QUADRANTS:                 # drop a quadrant written in front
        words = words[1:]
    return ' '.join(words)

segments['road'] = segments['on_road'].apply(clean_name)
crashes['road'] = (crashes['StreetName'] + ' ' + crashes['StreetSuffix'].fillna('')).apply(clean_name)

# Pair every crash with every segment on the SAME road, then measure the distance
pairs = crashes[['AccidentNumber', 'road', 'crash_lat', 'crash_lon']].merge(
    segments[['loc_id', 'road', 'latitude', 'longitude']], on='road')
dlat_ft = (pairs['crash_lat'] - pairs['latitude']) * FEET_PER_DEGREE_LAT
dlon_ft = (pairs['crash_lon'] - pairs['longitude']) * FEET_PER_DEGREE_LAT * math.cos(math.radians(35.2))
pairs['feet'] = np.sqrt(dlat_ft ** 2 + dlon_ft ** 2)

# Keep the closest segment, but only if it is within half a mile (so each segment covers ~1 mile)
MAX_FEET = 2640
pairs = pairs[pairs['feet'] <= MAX_FEET].sort_values('feet').drop_duplicates('AccidentNumber')
print('Collisions matched to a segment:', len(pairs), 'of', len(crashes))

collision_counts = pairs.groupby('loc_id').size().rename('collisions')
segments = segments.drop(columns='collisions', errors='ignore').merge(collision_counts, on='loc_id', how='left')
segments['collisions'] = segments['collisions'].fillna(0).astype(int)

## Step 5: Assign segments to wards
`find_wards` returns every ward within ~300 ft (0.001°) of the count location. A segment on or right next to a ward boundary is therefore **counted in both wards**, as the instructions require. `.explode()` turns a list like `[4, 8]` into two rows.

In [ ]:
# ---------- Wards (a segment near a boundary counts in BOTH wards) ----------
import geopandas as gpd
from shapely.geometry import Point

wards = gpd.read_file(DATA_FOLDER + 'Council_Wards_4934068591990916100.geojson')[['WARDNO', 'DIST_NAME', 'geometry']]

def find_wards(lat, lon):
    """Every ward within ~300 ft of the point (0.001 degrees)."""
    p = Point(lon, lat)
    return [wards['WARDNO'].iloc[i] for i in range(len(wards)) if wards['geometry'].iloc[i].distance(p) < 0.001]

segments['ward'] = segments.apply(lambda row: find_wards(row['latitude'], row['longitude']), axis=1)
outside = segments['ward'].str.len() == 0
print('Segments outside all wards (dropped):', outside.sum())
segments = segments[~outside].explode('ward')                    # one row per (segment, ward)
segments['ward'] = segments['ward'].astype(int)
print('Segments counted in two wards:', segments['loc_id'].duplicated().sum())

## Step 6: The rankings for each ward
1. **V/C rank:** highest V/C = 1
2. **Collision rank:** most collisions = 1
3. **Aggregate (rank-sum):** using only segments that have both volume and collision data, add the two ranks. **The smallest sum is the top priority.** Example from the instructions: 2nd in collisions + 4th in V/C = 6.

`groupby('ward')` makes each ranking separate for each ward, giving 8 lists. `method='min'` gives ties the same rank. Everything is saved to `Norman_Ward_Rankings.xlsx`, one sheet per ward, because the City already works in Excel.

In [ ]:
# ---------- The three rankings, separately for each ward ----------
# 1. V/C ranking (1 = most congested)
segments['vc_rank'] = segments.groupby('ward')['vc'].rank(ascending=False, method='min').astype(int)

# 2. Collision ranking (1 = most collisions), only segments that had collisions
has_crash = segments['collisions'] > 0
segments.loc[has_crash, 'collision_rank'] = segments[has_crash].groupby('ward')['collisions'].rank(ascending=False, method='min')

# 3. Aggregate: only segments with BOTH data -> add the two ranks -> smallest sum = top priority
both = segments[has_crash].copy()
both['vc_rank_both'] = both.groupby('ward')['vc'].rank(ascending=False, method='min')
both['combined'] = both['vc_rank_both'] + both['collision_rank']
both['priority_rank'] = both.groupby('ward')['combined'].rank(method='min').astype(int)
segments = segments.merge(both[['loc_id', 'ward', 'combined', 'priority_rank']], on=['loc_id', 'ward'], how='left')

segments['segment'] = segments['on_road'] + ' (' + segments['from'].fillna('?') + ' to ' + segments['to'].fillna('?') + ')'
show = ['segment', 'vpd', 'facility', 'vc', 'vc_rank', 'collisions', 'collision_rank', 'combined', 'priority_rank']

# Print the 8 priority lists (top 5 each)
for w in range(1, 9):
    print('\n=== Ward', w, '===')
    print(segments[segments['ward'] == w].sort_values('priority_rank')[show].head(5).to_string(index=False))

# Save everything to Excel: one sheet per ward (the city already uses Excel)
with pd.ExcelWriter(DATA_FOLDER + 'Norman_Ward_Rankings.xlsx') as writer:
    for w in range(1, 9):
        segments[segments['ward'] == w].sort_values(['priority_rank', 'vc_rank'])[show].to_excel(writer, sheet_name='Ward ' + str(w), index=False)

## Step 7: Maps over the City Ward Map (my assigned question)
1. **Collision heat map:** hexagon bins colored by how many collisions happened there
2. **Congestion map:** each count location colored by V/C
3. **Priority map:** each segment colored and sized by its rank-sum priority **within its own ward**, so every ward shows its own top projects

This addresses the City's concern about representation: each ward gets its own ranked list instead of one citywide top 20 that could be concentrated in a few wards.

In [ ]:
# ---------- Maps over the ward map ----------
import matplotlib.pyplot as plt

def ward_base_map(title):
    """Draw the ward map with labels and return the axes to draw on."""
    fig, ax = plt.subplots(figsize=(11, 8))
    wards.plot(ax=ax, column='WARDNO', categorical=True, cmap='Pastel1', edgecolor='black', alpha=0.6)
    for i in range(len(wards)):
        c = wards['geometry'].iloc[i].representative_point()
        ax.text(c.x, c.y, 'Ward ' + str(wards['WARDNO'].iloc[i]), ha='center', fontsize=11, weight='bold', alpha=0.7)
    ax.set_title(title, fontsize=14)
    ax.set_xticks([]); ax.set_yticks([])
    return fig, ax

one_row = segments.drop_duplicates('loc_id')     # boundary segments appear twice; draw them once

# Map 1: collision heat map
fig, ax = ward_base_map('Collision heat map, 2016-2026')
hb = ax.hexbin(crashes['crash_lon'], crashes['crash_lat'], gridsize=70, bins='log', cmap='YlOrRd', mincnt=1)
fig.colorbar(hb, ax=ax, shrink=0.7, label='Collisions (log scale)')
plt.savefig(DATA_FOLDER + 'map_collisions.png', dpi=150, bbox_inches='tight')
plt.show()

# Map 2: congestion (V/C)
fig, ax = ward_base_map('Congestion: volume-to-capacity ratio at each count location')
sc = ax.scatter(one_row['longitude'], one_row['latitude'], c=one_row['vc'], cmap='RdYlGn_r',
                s=60, edgecolor='black', linewidth=0.5, zorder=3)
fig.colorbar(sc, ax=ax, shrink=0.7, label='V/C (1.0 = at capacity)')
plt.savefig(DATA_FOLDER + 'map_vc.png', dpi=150, bbox_inches='tight')
plt.show()

# Map 3: aggregate priority (rank-sum), bigger + redder = higher priority in its ward
fig, ax = ward_base_map('Aggregate priority by ward (darker + bigger = higher priority in that ward)')
ranked = segments.dropna(subset=['priority_rank']).sort_values('priority_rank', ascending=False)
sc = ax.scatter(ranked['longitude'], ranked['latitude'], c=ranked['priority_rank'], cmap='Reds_r',
                s=200 / ranked['priority_rank'] + 15, edgecolor='black', linewidth=0.5, zorder=3, vmax=10)
fig.colorbar(sc, ax=ax, shrink=0.7, label='Priority rank within ward (1 = top)')
plt.savefig(DATA_FOLDER + 'map_priority.png', dpi=150, bbox_inches='tight')
plt.show()

# Business Question

# A.I. Tutor Use
I used the A.I. to help me debug my code, figure out specific syntax, and write regex.